# OLTW-Global

This notebook provides wrapper functions for calling our own implementation of OLTW-Global, where it is a variation of the OLTW algorithm by Simon Dixon. This notebook implements the `offline_processing()` and `online_processing()` function, which will be imported and run in `02_RunExperiment.ipynb`.


Here is a summary of the OLTW-Global approach:
- Offline processing: Chroma features are extracted from the piano reference audio and cached. The piano reference audio is also chopped to contain only the region of interest.
- Online processing: The query piano and reference piano recordings are aligned using the offline OLTW algorithm, which computes the full cost matrix before alignment. The alignment outputs frame indices that are then converted to seconds and adjusted to account for the reference offset.


## Offline Processing


In the offline processing stage, we:
1. Extract and cache chroma features from the piano reference audio
2. Chop the piano reference audio to contain only the region of interest

These cached features will be used during online processing for alignment computation.


In [5]:
from __future__ import annotations
import numpy as np
import librosa as lb
import os
import os.path
import system_utils
from scipy.io.wavfile import write
from typing import Callable


In [6]:
def offline_processing(scenario_dir, cache_dir, hop_length):
    '''
    Carries out offline processing for the OLTW-Global system by extracting and caching
    chroma features from the piano reference audio and chopping the reference audio.
    
    Inputs
    scenario_dir: The scenario directory to process
    cache_dir: The location of the cache directory
    hop_length: The hop length in samples used when computing chroma features
    
    This function will create cached chroma features and a chopped reference audio file.
    '''
    # Verify scenario directory
    system_utils.verify_scenario_dir(scenario_dir)
    
    # Create cache directory if it doesn't exist
    if not os.path.exists(cache_dir):
        os.makedirs(cache_dir)
    
    # Check if cache already exists
    cache_file = os.path.join(cache_dir, 'pref_stft.npy')
    if os.path.exists(cache_file):
        return
    
    # Get piano reference boundaries
    pref_path = os.path.join(scenario_dir, "pref.wav")
    if not os.path.exists(pref_path):
        raise FileNotFoundError(f"pref.wav missing in {scenario_dir}")
    
    try:
        p_start_t, p_end_t = system_utils.get_piano_reference_boundaries(scenario_dir)
    except (AssertionError, FileNotFoundError, ValueError) as e:
        raise ValueError(f"Cannot find piano reference boundaries in scenario.info: {e}")
    
    # Load and extract chroma features from full reference
    y_ref, sr_ref = lb.load(pref_path, sr=None)
    F_pref = lb.feature.chroma_stft(y=y_ref, sr=sr_ref, hop_length=hop_length, center=False)
    
    # Save chroma features to cache
    np.save(cache_file, F_pref)
    
    # Also chop the reference audio and save it
    y_chopped = y_ref[int(p_start_t * sr_ref): int(p_end_t * sr_ref)]
    out_path = os.path.join(scenario_dir, "pref_chopped.wav")
    write(out_path, sr_ref, (y_chopped * 32767).astype("int16"))  # 16-bit PCM
    
    return


In [7]:
def verify_cache_dir(indir):
    '''
    Verifies that the specified cache directory has the required files.
    
    Inputs
    indir: The cache directory to verify
    '''
    assert os.path.exists(os.path.join(indir, 'pref_stft.npy')), f'pref_stft.npy missing from {indir}'


## Online Processing



In the online processing stage, we:
1. Load the cached reference chroma features and extract query chroma features
2. Run the offline OLTW algorithm to compute the alignment path
3. Convert frame indices to seconds and adjust for reference offset
4. Save the alignment result


### Software Requirements

The OLTW-Global implementation uses the OnlineAlignment package. Ensure that the necessary modules can be imported from the OnlineAlignment folder.


In [ ]:
# Import OLTW-Global from OnlineAlignment
# We'll import these inside the function to avoid import errors if OnlineAlignment is not available
def _import_oltw_modules():
    '''
    Imports the necessary OLTW modules from OnlineAlignment.
    Returns a tuple of (run_offline_oltw, OLTW_STEPS, OLTW_WEIGHTS)
    '''
    try:
        from OnlineAlignment.core.alignment.offline.oltw import run_offline_oltw
        from OnlineAlignment.core.constants import OLTW_STEPS, OLTW_WEIGHTS
        return run_offline_oltw, OLTW_STEPS, OLTW_WEIGHTS
    except ImportError as e:
        raise ImportError(
            f"Failed to import OLTW modules from OnlineAlignment: {e}\n"
            "Please ensure the OnlineAlignment package is properly installed and accessible."
        )


In [ ]:
from noa import compute_cosine_distance, compute_euclidean_distance

def online_processing(scenario_dir, out_dir, cache_dir, hop_length,
                      DTW_steps=None, DTW_weights=None, 
                      window_steps=None, max_run_count=3, c=0,
                      cost_metric='cosine'):
    '''
    Carries out online processing using the offline OLTW algorithm.
    
    Inputs
    scenario_dir: The scenario directory to process
    out_dir: The directory to put results, intermediate files, and logging info
    cache_dir: The cache directory containing precomputed reference features
    hop_length: The hop length in samples used when computing chroma features
    DTW_steps: DTW window steps for cost matrix computation. Shape (n_steps, 2)
               If None, uses default OLTW_STEPS
    DTW_weights: DTW window weights for cost matrix computation. Shape (n_steps,)
                 If None, uses default OLTW_WEIGHTS
    window_steps: OLTW transition steps for path computation. Shape (3, 2)
                  If None, uses default OLTW_STEPS
    max_run_count: Maximum consecutive steps in one direction. Defaults to 3
    c: Band size for comparing costs. If None, no banding is used
    cost_metric: Cost metric to use ('cosine' or 'euclidean'). Defaults to 'cosine'

    This function will compute and save the predicted alignment in the output directory 
    in a file hyp.npy. The alignment is saved in seconds, with reference times relative 
    to the full reference audio.
    '''
    
    # Import OLTW modules
    run_offline_oltw, OLTW_STEPS, OLTW_WEIGHTS = _import_oltw_modules()
    
    # Set default steps and weights if not provided
    if DTW_steps is None:
        DTW_steps = OLTW_STEPS
    else:
        DTW_steps = np.array(DTW_steps)
    
    if DTW_weights is None:
        DTW_weights = OLTW_WEIGHTS
    else:
        DTW_weights = np.array(DTW_weights)
    
    if window_steps is None:
        window_steps = OLTW_STEPS
    else:
        window_steps = np.array(window_steps)
    
    # Verify & setup
    system_utils.verify_scenario_dir(scenario_dir)
    verify_cache_dir(cache_dir)
    assert not os.path.exists(out_dir), f'Output directory {out_dir} already exists.'
    os.makedirs(out_dir)
    
    # Get piano reference boundaries for offset calculation
    pref_start_sec, pref_end_sec = system_utils.get_piano_reference_boundaries(scenario_dir)
    
    # Load query features
    p_file = os.path.join(scenario_dir, 'p.wav')
    y_query, sr_query = lb.load(p_file, sr=None)
    F_query = lb.feature.chroma_stft(y=y_query, sr=sr_query, hop_length=hop_length, center=False)
    
    # Load reference features from cache
    F_ref = np.load(os.path.join(cache_dir, 'pref_stft.npy'))
    
    # Run offline OLTW alignment
    # Note: run_offline_oltw returns alignment path with shape (2, n_points)
    # where first row is query frame indices and second row is reference frame indices

    print("Running offline OLTW alignment...")

    alignment_path = run_offline_oltw(
        reference_features=F_ref,
        query_features=F_query,
        DTW_steps=DTW_steps,
        DTW_weights=DTW_weights,
        window_steps=window_steps,
        cost_metric=cost_metric,
        max_run_count=max_run_count,
        c=c,
        use_parallel_cost=True
    )

    print("Alignment completed. Processing results...")
    
    # Convert frame indices to seconds
    # alignment_path has shape (2, n_points) where:
    # - row 0: query frame indices
    # - row 1: reference frame indices
    hop_sec = hop_length / sr_query
    
    query_frames = alignment_path[0, :]
    ref_frames = alignment_path[1, :]
    
    query_seconds = query_frames * hop_sec
    # Convert reference frame times to seconds and adjust for reference offset in full audio
    ref_seconds = ref_frames * hop_sec + pref_start_sec
    
    # Create output alignment array in format (2, n_points)
    alignment_array = np.array([query_seconds, ref_seconds])
    
    # Save alignment
    np.save(os.path.join(out_dir, 'hyp.npy'), alignment_array)
    
    return


In [10]:
def verify_hyp_dir(indir):
    '''
    Verifies that the specified scenario hypothesis directory has the required files.
    
    Inputs
    indir: The hypothesis directory to verify
    '''
    assert os.path.exists(os.path.join(indir, 'hyp.npy')), f'{indir} is missing hyp.npy, please re-run online processing'
